# FLUX.1-dev GGUF + PuLID — Gate G1 (Task 1.1)
Measures **peak VRAM** and **time per image** for `Q4_0` (and `Q8_0`) with PuLID on/off, under 2 GPU layouts, then **fixes the standard generation configuration**.

| Section | Step | Notes |
|---|---|---|
| 1 | Install libraries (run once, auto-restarts) | |
| 2 | Shared configuration | **Edit parameters here only** |
| 3 | Load FLUX GGUF | change `QUANT` to switch Q4 ↔ Q8 |
| 4 | Load PuLID + hooks | |
| 5 | Face-similarity tool (insightface) | |
| 6 | Shared functions (layout, encode, `one_run`) | foundation for `generate()` in Task 1.3 |
| 7 | Smoke test | must show `CA fired = 240` |
| 8 | **G1: measure the matrix** | Task 1.1 |
| 9 | **G1: fix the standard configuration** | Task 1.1 |
| 10 | Appendix: multi-context test | optional |
| 11 | Export results / push to GitHub | |

**Before running:** (1) create the Kaggle Secret `HF_TOKEN` (and `GH_TOKEN` if you want to push); (2) upload the reference face image as a dataset and edit `REF_SRC` in section 2; (3) enable 2 T4 GPUs. **Run cells in order, top to bottom.**

## 1. Install libraries *(run once per session; the kernel restarts itself)*

In [ ]:
!pip install -q -U transformers diffusers accelerate "gguf>=0.10.0" "bitsandbytes>=0.46.1" \
    insightface onnxruntime facexlib ftfy timm einops torchsde open_clip_torch pyyaml
!git clone -q https://github.com/ToTheBeginning/PuLID.git /kaggle/working/PuLID 2>/dev/null || echo "PuLID already present"

import os, time
print("🔄 Kernel restarts in 3 seconds. Then continue from section 2.")
time.sleep(3)
os._exit(0)

## 2. Shared configuration
**All parameters live here.** Set `QUANT = "Q8_0"` and restart the kernel to measure Q8.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"   # must be set BEFORE importing torch

import gc, json, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

# ---------- Model ----------
QUANT      = "Q4_0"                       # "Q4_0" or "Q8_0"
BASE_PATH  = "/kaggle/input/datasets/YOUR_KAGGLE_USER/flux-vision-models"
GGUF_PATHS = {
    "Q4_0": f"{BASE_PATH}/flux_gguf/flux1-dev-Q4_0.gguf",
    "Q8_0": "/kaggle/input/datasets/YOUR_KAGGLE_USER/flux1-dev-q8-0-gguf/flux1-dev-Q8_0.gguf",  # double-check this path
}
TEXT_DIR   = f"{BASE_PATH}/vae_text_encoder"
PULID_CKPT = f"{BASE_PATH}/pulid/pulid_flux_v0.9.1.safetensors"
PULID_REPO = "/kaggle/working/PuLID"

# ---------- Reference image (upload as a Kaggle Dataset; do NOT download from a website) ----------
REF_SRC    = "/kaggle/input/datasets/YOUR_KAGGLE_USER/ref-face/ref_face.jpg"
REF_IMAGE  = "/kaggle/working/ref_face.jpg"

# ---------- Devices ----------
FLUX_DEV    = "cuda:0"          # transformer, VAE, pulid_ca
TXT_DEV     = "cuda:1"          # T5 + CLIP khi layout = "split"
ID_DEV      = "cuda:1"          # encoder PuLID / EVA-CLIP
T5_LOAD_DEV = FLUX_DEV          # if moving the 4-bit T5 with .to() fails: set to TXT_DEV and restart
DTYPE       = torch.float16     # T4 has poor bf16 support

# ---------- Generation ----------
PROMPT = ("A cinematic portrait of a Vietnamese woman wearing traditional ao dai, "
          "lotus garden background, soft natural lighting, highly detailed, photorealistic, 8k")
STEPS, GUIDANCE, ID_WEIGHT = 20, 3.5, 1.0
SEED0 = 42

# ---------- Gate G1 ----------
SIZES         = [(720, 720), (1024, 1024)]   # 1024 may OOM on a T4 -> it is recorded
N_MEASURED    = 3          # measured runs per configuration (quick test: set 1)
WARMUP_STEPS  = 4
VRAM_LIMIT_GB = 14.0       # T4 has ~15 GB; leave room for the CUDA context
MAX_TIME_S    = 180        # time-per-image budget when choosing the configuration

RESULTS_DIR = Path("/kaggle/working/results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    # Read the token from Kaggle Secrets (or an environment variable). NEVER write a token into code.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

assert torch.cuda.device_count() >= 2, "Two GPUs required (Kaggle T4 x2)"
print("GPU:", [torch.cuda.get_device_name(i) for i in range(2)], "| QUANT =", QUANT)

## 3. Load FLUX GGUF
The 4-bit T5-xxl, CLIP, VAE and the GGUF transformer are assembled into a pipeline by hand.

In [ ]:
import gc, torch; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from diffusers import (FluxPipeline, FluxTransformer2DModel, AutoencoderKL,
                       FlowMatchEulerDiscreteScheduler, GGUFQuantizationConfig)
from transformers import (T5EncoderModel, CLIPTextModel, CLIPTokenizer,
                          T5TokenizerFast, BitsAndBytesConfig)

FLUX_CONFIG = {
    "_class_name": "FluxTransformer2DModel", "_diffusers_version": "0.30.0",
    "attention_head_dim": 128, "guidance_embeds": True, "in_channels": 64,
    "joint_attention_dim": 4096, "num_attention_heads": 24, "num_layers": 19,
    "num_single_layers": 38, "patch_size": 1, "pooled_projection_dim": 768,
    "axes_dims_rope": [16, 56, 56],
}

class FluxPipelineGPU0(FluxPipeline):
    # Always executes on FLUX_DEV even if the text encoders live on another GPU.
    @property
    def _execution_device(self):
        return torch.device(FLUX_DEV)

def load_tokenizers():
    tok_dir = Path("/kaggle/working/flux_tokenizers")     # saved so the token is not needed next time
    if (tok_dir / "tokenizer").is_dir():
        return (CLIPTokenizer.from_pretrained(str(tok_dir / "tokenizer")),
                T5TokenizerFast.from_pretrained(str(tok_dir / "tokenizer_2")))
    token = get_secret("HF_TOKEN")
    assert token, "Missing HF_TOKEN: add a Kaggle Secret named HF_TOKEN"
    t1 = CLIPTokenizer.from_pretrained("black-forest-labs/FLUX.1-dev", subfolder="tokenizer", token=token)
    t2 = T5TokenizerFast.from_pretrained("black-forest-labs/FLUX.1-dev", subfolder="tokenizer_2", token=token)
    t1.save_pretrained(str(tok_dir / "tokenizer")); t2.save_pretrained(str(tok_dir / "tokenizer_2"))
    return t1, t2

def load_flux(quant):
    gguf = GGUF_PATHS[quant]
    assert Path(gguf).exists(), f"GGUF file not found: {gguf}"
    cfg_dir = Path("/kaggle/working/flux_transformer_config"); cfg_dir.mkdir(exist_ok=True)
    (cfg_dir / "config.json").write_text(json.dumps(FLUX_CONFIG, indent=2))
    gc.collect(); torch.cuda.empty_cache()

    transformer = FluxTransformer2DModel.from_single_file(
        gguf, config=str(cfg_dir),
        quantization_config=GGUFQuantizationConfig(compute_dtype=DTYPE),
        torch_dtype=DTYPE, local_files_only=True).to(FLUX_DEV)
    t5 = T5EncoderModel.from_pretrained(
        f"{TEXT_DIR}/text_encoder_2",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE),
        torch_dtype=DTYPE, local_files_only=True,
        device_map={"": torch.device(T5_LOAD_DEV).index})
    clip = CLIPTextModel.from_pretrained(f"{TEXT_DIR}/text_encoder", torch_dtype=DTYPE,
                                         local_files_only=True).to(FLUX_DEV)
    vae = AutoencoderKL.from_pretrained(f"{TEXT_DIR}/vae", torch_dtype=DTYPE,
                                        local_files_only=True).to(FLUX_DEV)
    tok1, tok2 = load_tokenizers()
    pipe = FluxPipelineGPU0(scheduler=FlowMatchEulerDiscreteScheduler(), vae=vae,
                            text_encoder=clip, text_encoder_2=t5,
                            tokenizer=tok1, tokenizer_2=tok2, transformer=transformer)
    pipe.vae.enable_tiling(); pipe.vae.enable_slicing()
    gc.collect(); torch.cuda.empty_cache()
    return pipe

def mem(tag=""):
    for d in (0, 1):
        free, total = torch.cuda.mem_get_info(d)
        print(f"{tag} GPU{d}: free {free/1e9:.2f}/{total/1e9:.2f} GB | allocated {torch.cuda.memory_allocated(d)/1e9:.2f} GB")

print(f"⏳ Loading FLUX {QUANT} ...")
pipe = load_flux(QUANT)
mem("[after loading FLUX]")

## 4. Load PuLID, ID embedding and hooks
The hooks add PuLID cross-attention to 20 transformer blocks (10 double + 10 single).

In [ ]:
# ---- 4.1 Reference image ----
assert "<" not in REF_SRC, "Set REF_SRC (section 2) to the path of the real image dataset"
shutil.copy(REF_SRC, REF_IMAGE)
ref_img = Image.open(REF_IMAGE).convert("RGB")
print("Reference image:", ref_img.size)
display(ref_img.resize((200, int(200 * ref_img.height / ref_img.width))))

In [ ]:
# ---- 4.2 Load PuLID ----
from safetensors.torch import load_file
sys.path.insert(0, PULID_REPO)
os.chdir("/kaggle/working")          # the PuLID repo downloads antelopev2 / EVA-CLIP into ./models

def load_pulid(pipe):
    from pulid.pipeline_flux import PuLIDPipeline
    ckpt = PULID_CKPT
    if not Path(ckpt).exists():
        from huggingface_hub import hf_hub_download
        ckpt = hf_hub_download("guozinan/PuLID", "pulid_flux_v0.9.1.safetensors",
                               local_dir="/kaggle/working/models")
    # ONNX runs on CPU so it does not use VRAM
    pulid = PuLIDPipeline(pipe.transformer, ID_DEV, weight_dtype=torch.float16, onnx_provider="cpu")
    sd = load_file(ckpt)
    groups = {}
    for k, v in sd.items():                       # prefixes: pulid_encoder.* and pulid_ca.*
        m, rest = k.split(".", 1)
        groups.setdefault(m, {})[rest] = v
    for m, s in groups.items():
        print(m, "->", getattr(pulid, m).load_state_dict(s, strict=True))
    del sd, groups; gc.collect()
    pulid.pulid_ca.to(FLUX_DEV).eval()            # CA lives on the same GPU as the transformer
    pulid.pulid_encoder.eval()
    return pulid

def make_id_embedding(pulid, img):
    with torch.no_grad():
        id_emb, _ = pulid.get_id_embedding(np.array(img), cal_uncond=False)
    assert not torch.isnan(id_emb).any(), "ID embedding is NaN (fp16 overflow?)"
    return id_emb.to(FLUX_DEV, DTYPE)

if "pulid" not in globals():
    pulid = load_pulid(pipe)
ID_EMB = make_id_embedding(pulid, ref_img)
N_CA = len(pulid.pulid_ca)
print("ID embedding:", tuple(ID_EMB.shape), ID_EMB.dtype, ID_EMB.device, "| CA count:", N_CA)

In [ ]:
# ---- 4.3 Hook ----
STATE = {"id": None, "weight": ID_WEIGHT, "txt_len": 512, "fired": 0}
HOOKS = globals().get("HOOKS", [])

def install_pulid_hooks(transformer, pulid):
    for h in HOOKS:                     # remove old hooks if the cell is re-run
        h.remove()
    HOOKS.clear()

    def apply_ca(idx, img_tokens):
        STATE["fired"] += 1
        return img_tokens + STATE["weight"] * pulid.pulid_ca[idx](STATE["id"], img_tokens)

    def double_hook(idx):
        def hook(module, args, output):
            if STATE["id"] is None:
                return output
            enc, hid = output
            return enc, apply_ca(idx, hid)
        return hook

    def single_hook(idx):
        def hook(module, args, output):
            if STATE["id"] is None:
                return output
            if isinstance(output, tuple):               # newer diffusers: (enc, hid)
                enc, hid = output
                return enc, apply_ca(idx, hid)
            n = STATE["txt_len"]                        # older diffusers: tensor [txt | img]
            return torch.cat([output[:, :n], apply_ca(idx, output[:, n:])], dim=1)
        return hook

    def pre_hook(module, args, kwargs):                 # capture the text token length
        e = kwargs.get("encoder_hidden_states", args[1] if len(args) > 1 else None)
        if e is not None:
            STATE["txt_len"] = e.shape[1]

    HOOKS.append(transformer.register_forward_pre_hook(pre_hook, with_kwargs=True))
    for i, blk in enumerate(transformer.transformer_blocks):          # 19 double -> 10 CA
        if i % 2 == 0:
            HOOKS.append(blk.register_forward_hook(double_hook(i // 2)))
    n_double = len(HOOKS) - 1
    for j, blk in enumerate(transformer.single_transformer_blocks):   # 38 single -> 10 CA
        if j % 4 == 0:
            HOOKS.append(blk.register_forward_hook(single_hook(n_double + j // 4)))
    n_hooks = len(HOOKS) - 1
    assert n_hooks == len(pulid.pulid_ca), f"Hook count {n_hooks} != CA count {len(pulid.pulid_ca)}"
    return n_hooks

print("Hook CA:", install_pulid_hooks(pipe.transformer, pulid), "(expected 20)")

## 5. Face-similarity tool
Used for a quick check in G1. The project's official metric (AdaFace/FaceNet) is implemented in Task 2.3.

In [ ]:
import cv2
from insightface.app import FaceAnalysis

face_app = FaceAnalysis(name="antelopev2", root="/kaggle/working", providers=["CPUExecutionProvider"])
face_app.prepare(ctx_id=0, det_size=(640, 640))

def _largest(faces):
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])) if faces else None

def face_of(img, app=face_app):
    return _largest(app.get(cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)))

REF_FACE = face_of(ref_img)
assert REF_FACE is not None, "❌ No face detected in the reference image"

def id_similarity(img):
    f = face_of(img)
    return None if f is None else float(np.dot(REF_FACE.normed_embedding, f.normed_embedding))

## 6. Shared functions
`set_layout` chooses where the text encoders live; `one_run` generates one image and returns timing + VRAM. This is also the core that becomes `generate()` in Task 1.3.

In [ ]:
LAYOUTS = {"1GPU": FLUX_DEV, "split": TXT_DEV}      # location of T5 + CLIP

def text_device():
    return str(next(pipe.text_encoder_2.parameters()).device)

def set_layout(name):
    dev = LAYOUTS[name]
    if text_device() != dev:
        pipe.text_encoder_2.to(dev)                  # 4-bit T5 first (most failure-prone)
        pipe.text_encoder.to(dev)
        gc.collect(); torch.cuda.empty_cache()

def sync():
    for d in range(2): torch.cuda.synchronize(d)

def reset_peaks():
    for d in range(2): torch.cuda.reset_peak_memory_stats(d)

def peaks():
    gb = 1024 ** 3
    return ({f"gpu{d}_alloc": torch.cuda.max_memory_allocated(d) / gb for d in range(2)} |
            {f"gpu{d}_resv":  torch.cuda.max_memory_reserved(d) / gb for d in range(2)})

@torch.no_grad()
def encode(text):
    dev = text_device()
    with torch.cuda.device(torch.device(dev).index):      # bnb 4-bit needs the correct current device
        pe, ppe, _ = pipe.encode_prompt(prompt=text, prompt_2=text, device=dev, max_sequence_length=512)
    return pe.to(FLUX_DEV, DTYPE), ppe.to(FLUX_DEV, DTYPE)

def one_run(text, w, h, steps=STEPS, use_id=True, seed=SEED0, guidance=GUIDANCE, weight=ID_WEIGHT):
    # Returns: image, t_encode, t_denoise (denoise + VAE), number of CA calls, peak VRAM per GPU
    reset_peaks(); sync()
    t0 = time.perf_counter()
    pe, ppe = encode(text); sync()
    t_enc = time.perf_counter() - t0
    STATE.update(id=ID_EMB if use_id else None, weight=weight, fired=0)
    t1 = time.perf_counter()
    img = pipe(prompt_embeds=pe, pooled_prompt_embeds=ppe, height=h, width=w,
               num_inference_steps=steps, guidance_scale=guidance,
               generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    sync()
    return img, t_enc, time.perf_counter() - t1, STATE["fired"], peaks()

## 7. Smoke test
Checks that PuLID really runs: `use_id=True` must give `CA fired = steps × CA count = 240`; `use_id=False` must give 0.

In [ ]:
SMOKE_STEPS = 12
for use_id in (False, True):
    img, t_enc, t_den, fired, pk = one_run(PROMPT, 512, 512, steps=SMOKE_STEPS, use_id=use_id, guidance=3.0)
    expected = SMOKE_STEPS * N_CA if use_id else 0
    sim = id_similarity(img)
    print(f"use_id={use_id!s:<5} | CA fired={fired:>3} (expected {expected}) | "
          f"ID sim={'no face found' if sim is None else f'{sim:.3f}'} | {t_enc + t_den:.1f}s")
    assert fired == expected, "PuLID did not fire the expected number of times -> check the hooks in section 4.3"
    display(img.resize((300, 300)))

## 8. G1: measure the matrix *(Task 1.1)*
Matrix: **{PuLID off, on} × {1GPU, split} × size**, with a warmup and `N_MEASURED` measured runs.

- **1GPU**: transformer, T5, CLIP, VAE and `pulid_ca` are all on `cuda:0`.
- **split**: T5 + CLIP on `cuda:1`, everything else on `cuda:0`.
- The ID embedding is precomputed, so the PuLID encoder is not counted in G1.
- `t_encode` = prompt encoding; `t_denoise` = denoise + VAE decode. VRAM = `max_memory_allocated` (excludes ~0.3–0.5 GB of CUDA context).
- For a first test run, set `N_MEASURED = 1` in section 2.

In [ ]:
def run_g1():
    runs = []
    for layout in LAYOUTS:
        try:
            set_layout(layout)
        except Exception as e:
            print(f"⚠️ Skipping layout {layout}: could not move the 4-bit T5 ({e}).\n"
                  f"   -> Set T5_LOAD_DEV in section 2 for the layout you want, restart the kernel and run again.")
            continue
        print(f"\n===== Layout {layout} | text encoder @ {text_device()} =====")
        for (w, h) in SIZES:
            for use_id in (False, True):
                tag = f"{layout} {w}x{h} ID={use_id}"
                base = dict(layout=layout, w=w, h=h, use_id=use_id)
                gc.collect(); torch.cuda.empty_cache()
                try:                                                  # cheap warmup
                    one_run(PROMPT, w, h, steps=WARMUP_STEPS, use_id=use_id)
                except torch.cuda.OutOfMemoryError:
                    gc.collect(); torch.cuda.empty_cache()
                    runs.append(base | dict(run=0, status="OOM")); print(f"❌ OOM (warmup) {tag}"); continue
                for i in range(N_MEASURED):
                    gc.collect(); torch.cuda.empty_cache()
                    try:
                        img, t_enc, t_den, fired, pk = one_run(PROMPT, w, h, use_id=use_id, seed=SEED0 + i)
                    except torch.cuda.OutOfMemoryError:
                        gc.collect(); torch.cuda.empty_cache()
                        runs.append(base | dict(run=i + 1, status="OOM")); print(f"❌ OOM {tag} run{i+1}"); break
                    black = bool(np.asarray(img).mean() < 5)          # black/NaN image caused by fp16
                    fired_ok = (fired == STEPS * N_CA) if use_id else (fired == 0)
                    sim = id_similarity(img)
                    runs.append(base | dict(run=i + 1, t_encode=t_enc, t_denoise=t_den, fired_ok=fired_ok,
                                            id_sim=None if sim is None else round(sim, 4),
                                            status="BLACK" if black else "OK", **pk))
                    print(f"✓ {tag} run{i+1}: enc {t_enc:.1f}s | den {t_den:.1f}s | "
                          f"GPU0 {pk['gpu0_alloc']:.2f} | GPU1 {pk['gpu1_alloc']:.2f} GB | fired_ok={fired_ok}")
                    if i == 0:
                        img.save(RESULTS_DIR / f"g1_{layout}_{w}x{h}_id{int(use_id)}.png")
    return pd.DataFrame(runs)

def summarize_g1(runs_df):
    rows = []
    for (layout, w, h, use_id), g in runs_df.groupby(["layout", "w", "h", "use_id"]):
        ok = g[g["status"] == "OK"]
        base = dict(quant=QUANT, pulid=bool(use_id), layout=layout, res=f"{w}x{h}", steps=STEPS)
        if ok.empty:
            rows.append(base | dict(n=0, status=",".join(sorted(set(g["status"]))))); continue
        total = ok["t_encode"] + ok["t_denoise"]
        v0, v1 = ok["gpu0_alloc"].max(), ok["gpu1_alloc"].max()
        rows.append(base | dict(
            n=len(ok), time_mean_s=round(total.mean(), 2), time_std_s=round(total.std(ddof=0), 2),
            t_encode_s=round(ok["t_encode"].mean(), 2), t_denoise_s=round(ok["t_denoise"].mean(), 2),
            vram_gpu0_gb=round(v0, 2), vram_gpu1_gb=round(v1, 2),
            vram_peak_gb=round(max(v0, v1), 2), fits_limit=bool(max(v0, v1) <= VRAM_LIMIT_GB),
            id_sim_mean=None if ok["id_sim"].isna().all() else round(ok["id_sim"].mean(), 3),
            fired_ok=bool(ok["fired_ok"].all()), status="OK" if len(ok) == len(g) else "PARTIAL"))
    return pd.DataFrame(rows)

env = {"gpus": [torch.cuda.get_device_name(d) for d in range(2)], "torch": torch.__version__,
       "quant": QUANT, "dtype": str(DTYPE), "steps": STEPS, "guidance": GUIDANCE, "n_measured": N_MEASURED}
(RESULTS_DIR / "g1_env.json").write_text(json.dumps(env, indent=2))

runs_df = run_g1()
runs_df.to_csv(RESULTS_DIR / "g1_runs.csv", index=False)
g1 = summarize_g1(runs_df)
g1.to_csv(RESULTS_DIR / "g1_table.csv", index=False)
print("\n📊 BẢNG G1"); display(g1)

## 9. G1: fix the standard generation configuration *(Task 1.1)*
Suggested rule: PuLID on, all runs completed, PuLID really fired, fits in VRAM → pick the largest resolution with time ≤ `MAX_TIME_S`. Review the table, then decide.

In [ ]:
import yaml

def decide_g1(g1):
    d = g1.copy()
    d["pixels"] = d["res"].map(lambda s: int(s.split("x")[0]) * int(s.split("x")[1]))
    cand = d[(d["pulid"] == True) & (d["status"] == "OK") & (d["fits_limit"] == True) & (d["fired_ok"] == True)]
    print("Candidates (PuLID on, eligible):")
    print(cand[["layout", "res", "time_mean_s", "vram_gpu0_gb", "vram_gpu1_gb", "id_sim_mean"]]
          .to_string(index=False) if len(cand) else "  (none)")
    ok_time = cand[cand["time_mean_s"] <= MAX_TIME_S]
    if ok_time.empty:
        print(f"\n⚠️ No configuration reaches <= {MAX_TIME_S}s and fits in VRAM: lower the resolution/steps or relax MAX_TIME_S.")
        return None
    best = ok_time.sort_values(["pixels", "time_mean_s"], ascending=[False, True]).iloc[0]
    w, h = map(int, best["res"].split("x"))
    return {"g1_chosen": {
        "quant": str(best["quant"]), "layout": str(best["layout"]), "width": w, "height": h,
        "steps": STEPS, "guidance": GUIDANCE, "dtype": "float16",
        "measured": {"time_mean_s": float(best["time_mean_s"]),
                     "vram_gpu0_gb": float(best["vram_gpu0_gb"]), "vram_gpu1_gb": float(best["vram_gpu1_gb"]),
                     "id_sim_mean": None if pd.isna(best["id_sim_mean"]) else float(best["id_sim_mean"])}}}

chosen = decide_g1(g1)
if chosen:
    text = yaml.safe_dump(chosen, allow_unicode=True, sort_keys=False)
    (RESULTS_DIR / f"g1_chosen_{QUANT}.yaml").write_text(text, encoding="utf-8")
    print("\n✅ Proposed standard configuration:\n" + text)

## 10. Appendix: multi-context test *(optional)*
Checks that PuLID keeps the face across different prompts, also using the independent `buffalo_l` recognizer. Not required for G1.

In [ ]:
CTX_TESTS = [   # (name, prompt, counted in the average?)
    ("front_studio", "studio portrait of a woman, front view, neutral expression, softbox lighting, plain grey background, photorealistic", True),
    ("side_3q",      "a woman in three-quarter side view looking out a window, golden hour light, cinematic, photorealistic", True),
    ("laugh_night",  "a woman laughing on a busy Saigon street at night, neon lights, shallow depth of field, photorealistic", True),
    ("full_body",    "full body shot of a woman in a red ao dai walking across an old wooden bridge, morning mist, photorealistic", True),
    ("low_light",    "a woman reading a book in a cafe, dim warm lighting, close-up, photorealistic", True),
    ("watercolor",   "a woman in watercolor illustration style, soft pastel colors, garden background", False),  # painting: poor face recognition is expected
]
CTX_SEEDS, CTX_WEIGHTS, CTX_SIZE = [42, 123], [1.0], 720

try:
    face_app2 = FaceAnalysis(name="buffalo_l", root="/kaggle/working",
                             allowed_modules=["detection", "recognition"], providers=["CPUExecutionProvider"])
    face_app2.prepare(ctx_id=0, det_size=(640, 640))
except Exception as e:
    face_app2 = None; print("Could not load buffalo_l:", e)

def sim_buffalo(img):
    if face_app2 is None: return None
    a, b = face_of(ref_img, face_app2), face_of(img, face_app2)
    return None if a is None or b is None else float(np.dot(a.normed_embedding, b.normed_embedding))

rows = []
for name, text, counted in CTX_TESTS:
    for wt in CTX_WEIGHTS:
        for seed in CTX_SEEDS:
            gc.collect(); torch.cuda.empty_cache()
            try:
                img, t_enc, t_den, _, _ = one_run(text, CTX_SIZE, CTX_SIZE, use_id=True, seed=seed, weight=wt)
            except torch.cuda.OutOfMemoryError:
                rows.append({"Context": name, "Seed": seed, "Weight": wt, "Status": "OOM"}); continue
            img.save(RESULTS_DIR / f"ctx_{name}_w{wt}_s{seed}.png")
            s1, s2 = id_similarity(img), sim_buffalo(img)
            rows.append({"Context": name, "Seed": seed, "Weight": wt, "Counted": counted,
                         "ID Sim (antelopev2)": None if s1 is None else round(s1, 3),
                         "ID Sim (buffalo_l)": None if s2 is None else round(s2, 3),
                         "Time (s)": round(t_enc + t_den, 1), "Status": "OK"})
            print(rows[-1]); display(img.resize((240, 240)))

df_ctx = pd.DataFrame(rows); display(df_ctx)
ok = df_ctx[(df_ctx["Status"] == "OK") & (df_ctx["Counted"] == True)]
if len(ok):
    print(ok.groupby("Weight")[["ID Sim (antelopev2)", "ID Sim (buffalo_l)"]].mean().round(3))
df_ctx.to_csv(RESULTS_DIR / "ctx_report.csv", index=False)

## 11. Export results
A Kaggle session loses its files when it ends, so **always run this section** before shutting down.

In [ ]:
# ---- 11.1 Zip for download ----
from IPython.display import FileLink
zip_base = f"/kaggle/working/flux_g1_{QUANT}"
shutil.make_archive(zip_base, "zip", RESULTS_DIR)
print("✅ Results zipped:"); display(FileLink(f"flux_g1_{QUANT}.zip"))

In [ ]:
# ---- 11.2 (Optional) Push the numbers to GitHub: small files only (csv/json/yaml), no images ----
PUSH     = False
GH_REPO  = "<username>/flux-pulid"        # change to your real repo

def sh(cmd, secret=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out.replace(secret, "***") if secret else out)
    if r.returncode:
        raise RuntimeError("Command failed (see the log above)")

if PUSH:
    assert "<" not in GH_REPO, "Set GH_REPO before pushing"
    tok = get_secret("GH_TOKEN"); assert tok, "Missing Kaggle Secret GH_TOKEN"
    repo = "/kaggle/working/flux-pulid-repo"
    sh(f"rm -rf {repo} && git clone -q https://{tok}@github.com/{GH_REPO}.git {repo}", secret=tok)
    sh(f"mkdir -p {repo}/results/{QUANT} && cp {RESULTS_DIR}/*.csv {RESULTS_DIR}/*.json {RESULTS_DIR}/*.yaml {repo}/results/{QUANT}/ 2>/dev/null || true")
    sh(f'cd {repo} && git config user.email "kaggle@example.com" && git config user.name "kaggle-runner" && git add results')
    sh(f'cd {repo} && (git diff --cached --quiet || git commit -q -m "G1 {QUANT}: VRAM/time measurements") && git push -q', secret=tok)
else:
    print("PUSH = False -> skipped. Set PUSH = True (and GH_REPO) when you want to push to GitHub.")